In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, count, avg, round
from pyspark.sql.window import Window

# Criar o banco de dados Gold (se não existir)
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# ==============================================================================
# 1. CRIAÇÃO DAS DIMENSÕES (Tabelas de Domínio/Catálogo)
# ==============================================================================

# --- A) gold.dim_movies ---
df_movies_silver = spark.read.table("silver.tb_info_filmes")

df_dim_movies = df_movies_silver.select(
    "id_filme", "titulo", "data_lancamento", "ano_lancamento", 
    "duracao_minutos", "idioma_original", "status_filme", "sinopse"
).dropDuplicates(["id_filme"])

df_dim_movies = df_dim_movies.withColumn("sk_movie_id", monotonically_increasing_id())
df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")


# --- B) gold.dim_genres ---
df_generos_silver = spark.read.table("silver.tb_generos")

df_dim_genres = df_generos_silver.select(col("genero").alias("nome_genero")).dropDuplicates()
df_dim_genres = df_dim_genres.withColumn("sk_genre_id", monotonically_increasing_id())
df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")


# Lendo a tabela unificada de pessoas e empresas
df_pessoas_empresas = spark.read.table("silver.tb_pessoas_empresas")

# --- C) gold.dim_people ---
df_dim_people = df_pessoas_empresas.filter(col("tipo_entidade") != "Produtora") \
    .select(col("nome_entidade").alias("nome_pessoa"), col("tipo_entidade").alias("tipo_pessoa")) \
    .dropDuplicates()
df_dim_people = df_dim_people.withColumn("sk_person_id", monotonically_increasing_id())
df_dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")


# --- D) gold.dim_companies ---
df_dim_companies = df_pessoas_empresas.filter(col("tipo_entidade") == "Produtora") \
    .select(col("nome_entidade").alias("nome_produtora")).dropDuplicates()
df_dim_companies = df_dim_companies.withColumn("sk_company_id", monotonically_increasing_id())
df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")


# ==============================================================================
# 2. CRIAÇÃO DAS TABELAS-PONTE (Resolução de Relacionamentos N:N)
# ==============================================================================

dim_movies = spark.read.table("gold.dim_movies")
dim_genres = spark.read.table("gold.dim_genres")
dim_people = spark.read.table("gold.dim_people")
dim_companies = spark.read.table("gold.dim_companies")

# --- Bridge Movie Genre ---
df_bridge_genre = df_generos_silver.join(dim_movies, df_generos_silver.id_filme == dim_movies.id_filme, "inner") \
    .join(dim_genres, df_generos_silver.genero == dim_genres.nome_genero, "inner") \
    .select("sk_movie_id", "sk_genre_id").dropDuplicates()
df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")

# --- Bridge Movie Person ---
df_bridge_person = df_pessoas_empresas.filter(col("tipo_entidade") != "Produtora") \
    .join(dim_movies, df_pessoas_empresas.id_filme == dim_movies.id_filme, "inner") \
    .join(dim_people, (df_pessoas_empresas.nome_entidade == dim_people.nome_pessoa) & (df_pessoas_empresas.tipo_entidade == dim_people.tipo_pessoa), "inner") \
    .select("sk_movie_id", "sk_person_id").dropDuplicates()
df_bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")

# --- Bridge Movie Company ---
df_bridge_company = df_pessoas_empresas.filter(col("tipo_entidade") == "Produtora") \
    .join(dim_movies, df_pessoas_empresas.id_filme == dim_movies.id_filme, "inner") \
    .join(dim_companies, df_pessoas_empresas.nome_entidade == dim_companies.nome_produtora, "inner") \
    .select("sk_movie_id", "sk_company_id").dropDuplicates()
df_bridge_company.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_company")


# ==============================================================================
# 3. CRIAÇÃO DA DIMENSÃO DE AVALIAÇÕES E TABELA FATO
# ==============================================================================

# --- E) gold.dim_reviews ---
df_avaliacoes_silver = spark.read.table("silver.tb_avaliacoes_usuarios")

df_dim_reviews = df_avaliacoes_silver.groupBy("id_filme").agg(
    count("nota_usuario").alias("qtd_avaliacoes_usuarios"),
    round(avg("nota_usuario"), 2).alias("nota_media_usuarios")
)
df_dim_reviews = df_dim_reviews.join(dim_movies, ["id_filme"], "inner") \
    .select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios") \
    .withColumn("sk_review_id", monotonically_increasing_id())

df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")


# --- F) gold.fact_movies_performance ---
df_financeiro = spark.read.table("silver.tb_financeiro_filmes")
df_engajamento = spark.read.table("silver.tb_metricas_engajamento")

df_fact = df_financeiro.join(df_engajamento, ["id_filme"], "inner")
df_fact = df_fact.join(dim_movies, ["id_filme"], "inner")

df_fact = df_fact.select(
    "sk_movie_id",
    col("orcamento_usd").cast("decimal(18,2)"),
    col("receita_usd").cast("decimal(18,2)"),
    col("lucro_usd").cast("decimal(18,2)"),
    col("orcamento_brl").cast("decimal(18,2)"),
    col("receita_brl").cast("decimal(18,2)"),
    col("lucro_brl").cast("decimal(18,2)"),
    col("popularidade").cast("double"),
    col("nota_media_tmdb").cast("double"),
    col("qtd_votos_tmdb").cast("int"),
    col("nota_media_imdb").cast("double"),
    col("qtd_votos_imdb").cast("int")
).dropDuplicates(["sk_movie_id"]) 

df_fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")

print("Star Schema da Camada Gold gerado com sucesso!")

Star Schema da Camada Gold gerado com sucesso!


In [0]:
from pyspark.sql.functions import col, sum, count, rank, max, add_months, lit
from pyspark.sql.window import Window

# Carregando as tabelas da camada Gold
fato = spark.read.table("gold.fact_movies_performance")
dim_movies = spark.read.table("gold.dim_movies")
dim_genres = spark.read.table("gold.dim_genres")
bridge_genre = spark.read.table("gold.bridge_movie_genre")
dim_people = spark.read.table("gold.dim_people")
bridge_person = spark.read.table("gold.bridge_movie_person")
dim_companies = spark.read.table("gold.dim_companies")
bridge_company = spark.read.table("gold.bridge_movie_company")

print("--- 1. Qual é a receita total (em R$) somada de todos os filmes da base? ---")
q1 = fato.select(sum("receita_brl").alias("receita_total_brl"))
display(q1)


print("--- 2. Quais são os 5 filmes com maior popularidade? ---")
# Regra: Garantir que a popularidade não seja nula, filtrar ruídos e ordenar decrescente estrito.
q2 = fato.join(dim_movies, "sk_movie_id", "inner") \
         .filter((col("popularidade").isNotNull()) & (col("popularidade") <= 5000)) \
         .select("titulo", "popularidade") \
         .orderBy(col("popularidade").desc()) \
         .limit(5)
display(q2)


print("--- 3. Quantos filmes cada gênero possui? ---")
generos_oficiais = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", 
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery", 
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

q3 = dim_genres.join(bridge_genre, "sk_genre_id", "inner") \
               .filter(col("nome_genero").isin(generos_oficiais)) \
               .groupBy("nome_genero") \
               .agg(count("sk_movie_id").alias("quantidade_filmes")) \
               .orderBy(col("quantidade_filmes").desc())
display(q3)


print("--- 4. Ranking dos 10 filmes de maior receita (US$ e R$) ---")
janela_ranking = Window.orderBy(col("receita_usd").desc())

q4 = fato.join(dim_movies, "sk_movie_id", "inner") \
         .filter(col("receita_usd").isNotNull()) \
         .withColumn("posicao_ranking", rank().over(janela_ranking)) \
         .select("posicao_ranking", "titulo", "receita_usd", "receita_brl") \
         .orderBy("posicao_ranking") \
         .limit(10)
display(q4)


# Identificar a data máxima de lançamento na base
max_data_row = dim_movies.select(max("data_lancamento").alias("max_date")).first()
max_data_valida = max_data_row["max_date"]
max_data_lit = lit(max_data_valida)


print(f"--- 5. Qual ator teve a maior quantidade de participações nos últimos 2 anos (ou histórico geral se vazio)? ---")
# Tentativa com o filtro de 2 anos
q5 = dim_people.filter(col("tipo_pessoa") == "Ator") \
               .join(bridge_person, "sk_person_id", "inner") \
               .join(dim_movies, "sk_movie_id", "inner") \
               .filter(col("data_lancamento").between(add_months(max_data_lit, -24), max_data_lit)) \
               .groupBy("nome_pessoa") \
               .agg(count("sk_movie_id").alias("qtd_participacoes")) \
               .orderBy(col("qtd_participacoes").desc()) \
               .limit(1)

# Se o intervalo temporal restrito não retornar dados devido à antiguidade da base, buscamos no histórico geral
if q5.count() == 0:
    q5 = dim_people.filter(col("tipo_pessoa") == "Ator") \
                   .join(bridge_person, "sk_person_id", "inner") \
                   .groupBy("nome_pessoa") \
                   .agg(count("sk_movie_id").alias("qtd_participacoes")) \
                   .orderBy(col("qtd_participacoes").desc()) \
                   .limit(1)
display(q5)


print(f"--- 6. Qual a produtora com o maior Lucro nos últimos 5 anos (ou histórico geral)? ---")
q6 = dim_companies.join(bridge_company, "sk_company_id", "inner") \
                  .join(dim_movies, "sk_movie_id", "inner") \
                  .join(fato, "sk_movie_id", "inner") \
                  .filter((col("lucro_usd").isNotNull()) & (col("data_lancamento").between(add_months(max_data_lit, -60), max_data_lit))) \
                  .groupBy("nome_produtora") \
                  .agg(sum("lucro_usd").alias("lucro_total_usd")) \
                  .orderBy(col("lucro_total_usd").desc()) \
                  .limit(1)

if q6.count() == 0 or q6.first()["lucro_total_usd"] is None:
    q6 = dim_companies.join(bridge_company, "sk_company_id", "inner") \
                      .join(fato, "sk_movie_id", "inner") \
                      .filter(col("lucro_usd").isNotNull()) \
                      .groupBy("nome_produtora") \
                      .agg(sum("lucro_usd").alias("lucro_total_usd")) \
                      .orderBy(col("lucro_total_usd").desc()) \
                      .limit(1)
display(q6)

--- 1. Qual é a receita total (em R$) somada de todos os filmes da base? ---


receita_total_brl
811262462345.70


--- 2. Quais são os 5 filmes com maior popularidade? ---


titulo,popularidade
Doc & Darryl,3030.0
Purge of Kingdoms,3000.0
blue beetle,2994.357
Gran Turismo,2680.593
Obamaland Part 1: Rise of the Trumpublikans,2040.0


--- 3. Quantos filmes cada gênero possui? ---


...

posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13958840000.00
2,Avatar: The Way of Water,2320250281.00,11567143725.87
3,AVENGERS: INFINITY WAR,2052415039.00,10231904693.93
4,spider-man: no way home,1921847111.00,9580984402.47
5,The Lion King,1663075401.00,8290929796.61
6,Top Gun: Maverick,1488732821.00,7421779732.53
7,Barbie,1428545028.00,7121725528.09
8,The Super Mario Bros. Movie,1355725263.00,6758697153.63
9,Black Panther,1349926083.00,6729786501.58
10,Star Wars: The Last Jedi,1332698830.00,6643903477.20


--- 5. Qual ator teve a maior quantidade de participações nos últimos 2 anos (ou histórico geral se vazio)? ---


nome_pessoa,qtd_participacoes
Eric Roberts,194


--- 6. Qual a produtora com o maior Lucro nos últimos 5 anos (ou histórico geral)? ---


nome_produtora,lucro_total_usd
Universal Pictures,16276964878.00
